# Modelling the connectivity

In [2]:
# load packages and install r remotes of graphab and circuitscape
library("pacman")
p_load("graph4lg", 
      "circuitscaper", 
      "terra", 
      "sf", 
      "leastcostpath")

Installing package into ‘C:/Users/Lukas/AppData/Local/R/win-library/4.5’
(as ‘lib’ is unspecified)
also installing the dependencies ‘segmented’, ‘RcppParallel’, ‘seqinr’, ‘permute’, ‘gaston’, ‘gtools’, ‘adegenet’, ‘vegan’, ‘pegas’, ‘hierfstat’, ‘gdistance’, ‘ecodist’



  cannot open URL 'http://www.stats.ox.ac.uk/pub/RWin/bin/windows/contrib/4.5/PACKAGES'
trying URL 'https://cran.rstudio.com/bin/windows/contrib/4.5/segmented_2.2-1.zip'
trying URL 'https://cran.rstudio.com/bin/windows/contrib/4.5/RcppParallel_5.1.11-2.zip'
trying URL 'https://cran.rstudio.com/bin/windows/contrib/4.5/seqinr_4.2-36.zip'
trying URL 'https://cran.rstudio.com/bin/windows/contrib/4.5/permute_0.9-10.zip'
trying URL 'https://cran.rstudio.com/bin/windows/contrib/4.5/gaston_1.6.zip'
trying URL 'https://cran.rstudio.com/bin/windows/contrib/4.5/gtools_3.9.5.zip'
trying URL 'https://cran.rstudio.com/bin/windows/contrib/4.5/adegenet_2.1.11.zip'
trying URL 'https://cran.rstudio.com/bin/windows/contrib/4.5/vegan_2.7-3.zip'
trying URL 'https://cran.rstudio.com/bin/windows/contrib/4.5/pegas_1.4.zip'
trying URL 'https://cran.rstudio.com/bin/windows/contrib/4.5/hierfstat_0.5-11.zip'
trying URL 'https://cran.rstudio.com/bin/windows/contrib/4.5/gdistance_1.6.5.zip'
trying URL 'https://cran

package ‘segmented’ successfully unpacked and MD5 sums checked
package ‘RcppParallel’ successfully unpacked and MD5 sums checked
package ‘seqinr’ successfully unpacked and MD5 sums checked
package ‘permute’ successfully unpacked and MD5 sums checked
package ‘gaston’ successfully unpacked and MD5 sums checked
package ‘gtools’ successfully unpacked and MD5 sums checked
package ‘adegenet’ successfully unpacked and MD5 sums checked
package ‘vegan’ successfully unpacked and MD5 sums checked
package ‘pegas’ successfully unpacked and MD5 sums checked
package ‘hierfstat’ successfully unpacked and MD5 sums checked
package ‘gdistance’ successfully unpacked and MD5 sums checked
package ‘ecodist’ successfully unpacked and MD5 sums checked
package ‘graph4lg’ successfully unpacked and MD5 sums checked

The downloaded binary packages are in
	C:\Users\Lukas\AppData\Local\Temp\RtmpMXJ8Pp\downloaded_packages

graph4lg installed
Installing package into ‘C:/Users/Lukas/AppData/Local/R/win-library/4.5’
(as

  cannot open URL 'http://www.stats.ox.ac.uk/pub/RWin/bin/windows/contrib/4.5/PACKAGES'
trying URL 'https://cran.rstudio.com/bin/windows/contrib/4.5/rjson_0.2.23.zip'
trying URL 'https://cran.rstudio.com/bin/windows/contrib/4.5/JuliaCall_0.17.6.zip'
trying URL 'https://cran.rstudio.com/bin/windows/contrib/4.5/circuitscaper_0.1.0.zip'


package ‘rjson’ successfully unpacked and MD5 sums checked
package ‘JuliaCall’ successfully unpacked and MD5 sums checked
package ‘circuitscaper’ successfully unpacked and MD5 sums checked

The downloaded binary packages are in
	C:\Users\Lukas\AppData\Local\Temp\RtmpMXJ8Pp\downloaded_packages

circuitscaper installed


Warning messages:
1: package ‘graph4lg’ was built under R version 4.5.3 
2: package ‘circuitscaper’ was built under R version 4.5.3 


## Step 1: Landscape diagram (Graphab in R)
We use the graphab R package to automatically build the planar graph with the 30 km threshold in the background.

IMPROVE THIS CODE!

In [30]:
library(terra)
library(graph4lg)
library(sf)
library(ggplot2)

# --- PFADE DEFINIEREN ---
# Wir setzen das Arbeitsverzeichnis dorthin, wo das Projekt entstehen soll
root_dir <- "C:/ZHAW/6.Semester/BA/BA-wild-boar-connectivity-modeling"
setwd(file.path(root_dir, "data/processed"))

# Jetzt sind wir im 'data/processed' Ordner
input_raster <- "Core_Habitats_50ha_Nodes.tif"
proj_name <- "aargau_boar_final" 

print("Erstelle unkomprimierte TIF-Datei für Graphab 2.8...")

# 1. Raster laden
r_nodes <- rast(input_raster)

# 2. Ohne Kompression speichern (im aktuellen Verzeichnis)
pfad_sicher <- "Nodes_Uncompressed.tif"
writeRaster(r_nodes, pfad_sicher, datatype="INT2S", gdal=c("COMPRESS=NONE"), overwrite=TRUE)

# 3. Alten Ordner löschen
if(dir.exists(proj_name)) unlink(proj_name, recursive = TRUE)

print("Versuche Projekt-Erstellung erneut...")

# 4. Projekt erstellen (OHNE das 'dir' Argument)
# Graphab erstellt das Projekt jetzt automatisch im aktuellen Arbeitsverzeichnis
graphab_project(proj_name = proj_name, 
                raster = pfad_sicher, 
                habitat = 1,
                alloc_ram = 8)

# 5. Berechnungen durchführen
graphab_link(proj_name = proj_name, name = "links_planar", distance = "euclid", topo = "planar")
graphab_graph(proj_name = proj_name, linkset = "links_planar", name = "graph_30km", thr = 30000)
graphab_metric(proj_name = proj_name, graph = "graph_30km", metric = "BC", dist = 30000, prob = 0.05)

print(paste("Erfolg! Projekt wurde hier erstellt:", getwd(), "/", proj_name))

# --- TEIL 6: DATEN LADEN & PLOTTEN ---
nodes_path <- file.path(proj_name, "patches/patches.shp")
links_path <- file.path(proj_name, "links/links_planar-graph_30km.shp")

if(file.exists(nodes_path)) {
    nodes_sf <- st_read(nodes_path)
    links_sf <- st_read(links_path)
    
    p <- ggplot() +
      geom_sf(data = nodes_sf, aes(fill = BC), color = "white", size = 0.1) +
      geom_sf(data = links_sf, color = "red", size = 0.4, alpha = 0.6) +
      scale_fill_viridis_c(option = "magma") +
      theme_minimal() +
      labs(title = "Landschaftsdiagramm Wildschwein Aargau")
    
    print(p)
}

[1] "Erstelle unkomprimierte TIF-Datei für Graphab 2.8..."
[1] "Versuche Projekt-Erstellung erneut..."
Graphab project aargau_boar_final has been created in directory: C:\ZHAW\6.Semester\BA\BA-wild-boar-connectivity-modeling\data\processed
Link set 'links_planar' has been created in the project aargau_boar_final
Graph 'graph_30km' has been created in the project aargau_boar_final
Metric 'BC' has been computed in the project aargau_boar_final
[1] "Erfolg! Projekt wurde hier erstellt: C:/ZHAW/6.Semester/BA/BA-wild-boar-connectivity-modeling/data/processed / aargau_boar_final"


---

## Step 2: Circuit Theory Pinch-Points (Circuitscape / Omniscape)

We use circuitscaper (which runs Julia's Omniscape algorithm). Omniscape is literally the mathematical equivalent of the "Pelletier 2014 Tile Approach" mentioned in the paper—it automatically moves a window across your map to calculate omnidirectional current without crashing your RAM.

In [ ]:
library(JuliaCall)
library(circuitscaper)
library(terra)
library(ggplot2)
library(viridis)

print("Verbinde mit der sauberen Julia-Installation...")

# Wir verbieten R, eigene Downloads zu machen. 
# Da wir Julia zum PATH hinzugefügt haben, sollte R es jetzt magisch finden.
tryCatch({
  julia_setup(installJulia = FALSE)
  print("Julia erfolgreich verbunden!")
}, error = function(e) {
  print("R findet Julia noch nicht automatisch. Bitte gib den Pfad manuell an:")
  print("Führe stattdessen diesen Befehl aus: julia_setup(JULIA_HOME = 'C:/Users/Lukas/AppData/Local/Programs/Julia-1.10.3/bin')")
  # (Passe die Versionsnummer 1.10.3 an die an, die du gerade installiert hast)
})

# Wenn obiges geklappt hat, laden wir die Circuitscape-Pakete in Julia
print("Lade Omniscape-Module...")
cs_setup()

print("Bühne frei! Du kannst jetzt das os_run() Skript ausführen.")

Warning message:
package ‘JuliaCall’ was built under R version 4.5.3 
Warning message:
package ‘circuitscaper’ was built under R version 4.5.3 


terra 1.8.70


Warning message:
package ‘terra’ was built under R version 4.5.2 


Loading required package: viridisLite


Warning message:
package ‘viridis’ was built under R version 4.5.3 


[1] "Verbinde mit der sauberen Julia-Installation..."
Julia version 1.9.4 at location C:\Users\Lukas\AppData\Roaming\R\data\R\JULIAC~1\julia\19CB63~1.4\JULIA-~1.4\bin will be used.
Loading setup script for JuliaCall...
[1] "R findet Julia noch nicht automatisch. Bitte gib den Pfad manuell an:"
[1] "Führe stattdessen diesen Befehl aus: julia_setup(JULIA_HOME = 'C:/Users/Lukas/AppData/Local/Programs/Julia-1.10.3/bin')"
[1] "Lade Omniscape-Module..."
Initializing Julia (one-time per session)...


In [34]:
# ==============================================================================
# SCHRITT 2: KONNEKTIVITÄTSENGPÄSSE (Circuitscape / Omniscape)
# Umsetzung der Pelletier et al. 2014 Kachel-Methode
# ==============================================================================

library(terra)
library(circuitscaper)
library(ggplot2)
library(viridis)
p_load("JuliaCall")

# --- 1. PFADE SETZEN ---
base_path <- "C:/ZHAW/6.Semester/BA/BA-wild-boar-connectivity-modeling"
processed_dir <- file.path(base_path, "data/processed")
setwd(processed_dir)

# Dein fertig aufbereitetes Widerstandsraster (Passe den Namen an, falls er anders heisst!)
resistance_file <- "Resistance_Graphab_Ready.tif" 

# --- 2. JULIA & CIRCUITSCAPE SETUP ---
print("Verbinde R mit Julia (Circuitscape Engine)...")
# Falls Julia das erste Mal läuft, lade es herunter:
# cs_install_julia() 

# 1. Wir nehmen exakt den Pfad, in den R dein Julia gerade heruntergeladen hat 
# (ACHTUNG: Wir hängen '/bin' an, weil dort die Startdatei liegt)
julia_bin_pfad <- "C:/Users/Lukas/AppData/Roaming/R/data/R/JuliaCall/julia/1.9.4/julia-1.9.4/bin"

# 2. Wir zwingen R, die Kommunikations-Brücke neu zu kompilieren
julia_setup(JULIA_HOME = julia_bin_pfad, rebuild = TRUE)
cs_setup() # Startet die Julia-Umgebung im Hintergrund

# --- 3. BERECHNUNG DER OMNIDIREKTIONALEN STROMDICHTE ---
print("Lade Widerstandskarte...")
res_map <- rast(resistance_file)

print("Starte Omniscape (Automatisiert den Kachel-Ansatz nach Pelletier 2014)...")
print("ACHTUNG: Dies kann je nach Rechnerleistung einige Minuten bis Stunden dauern!")

# os_run simuliert den Strom. 'radius' entspricht der Kachelgrösse + Puffer.
# Ein Radius von 10000m (10 km) ist ein guter Standard für Wildschweine.
os_result <- os_run(resistance = res_map, 
                    radius = 10000, 
                    block_size = 5, # Speeds up calculation by grouping pixels
                    project_name = "boar_pinchpoints")

# --- 4. RESULTATE LADEN UND IN DEZILE EINTEILEN ---
print("Stromdichte berechnet! Teile Resultate in Dezile ein (wie im Paper gefordert)...")

# Lade die rohe Stromdichte-Karte
current_density <- rast("boar_pinchpoints/cum_currenc.tif")

# Ignoriere NoData und Nullen für die statistische Berechnung
vals <- values(current_density, mat = FALSE)
vals_valid <- vals[!is.na(vals) & vals > 0]

# Berechne die 10 Quantile (Dezile)
breaks <- quantile(vals_valid, probs = seq(0, 1, 0.1), na.rm = TRUE)

# Erstelle die Reklassifizierungs-Matrix
# Alle Werte zwischen Break 1 und Break 2 bekommen Wert 1, etc.
reclass_matrix <- cbind(breaks[-11], breaks[-1], 1:10)

# Raster reklassifizieren (1 = tiefe Bewegungswahrscheinlichkeit, 10 = absolute Engpässe)
current_deciles <- classify(current_density, reclass_matrix, include.lowest = TRUE)

# --- 5. EXPORT UND VISUALISIERUNG ---
output_tiff <- file.path(processed_dir, "Wildboar_PinchPoints_Deciles.tif")
writeRaster(current_deciles, output_tiff, overwrite = TRUE)

print(paste("ERFOLG: Dezil-Karte gespeichert unter:", output_tiff))

# Plot direkt im Notebook anzeigen
plot(current_deciles, 
     col = inferno(10), 
     main = "Konnektivitätsengpässe (Pinch-Points)\n1 = Unpassierbar | 10 = Kritischer Engpass",
     axes = FALSE)

[1] "Verbinde R mit Julia (Circuitscape Engine)..."
Julia version 1.9.4 at location C:\Users\Lukas\AppData\Roaming\R\data\R\JULIAC~1\julia\19CB63~1.4\JULIA-~1.4\bin will be used.
Loading setup script for JuliaCall...


: [1m[33mError[39m in `.julia$cmd()`:[22m
[33m![39m Error happens when you try to execute command ENV["R_HOME"] = "C:/Program Files/R/R-4.5.1";Base.include(Main,"C:/Users/Lukas/AppData/Local/R/win-library/4.5/JuliaCall/julia/setup.jl") in Julia.
                        To have more helpful error messages,
                        you could considering running the command in Julia directly